# 面试题：多 Agent 如何共享状态而不互相污染？

回答要点：共享的是有 schema、owner、version、可见范围的事件或 artifact，而不是任何 Agent 都能覆写的长文本。将事实、推测、计划和审批分层；事实追加写入、推测明确标记，读取按快照版本。下面用出差申请的六条黑板事件比较 last-write-wins 与 append-only 的可审计性。

## 真实案例

旅行规划、预算审核、合规三个 Agent 共同处理出差申请，事件包含权威预算、模型建议和审批证据。

## 基线

基线使用一个可被任意 Agent 覆盖的字典。

## 结果解读

手写黑板记录 owner、kind、version，按事实层读取。

## 失败案例

规划 Agent 的建议不能覆盖财务 Agent 的权威预算快照。

In [1]:
events = [{'agent':'planner','kind':'proposal','key':'city','value':'东京'}, {'agent':'finance','kind':'fact','key':'budget','value':800}, {'agent':'compliance','kind':'fact','key':'policy','value':'经济舱'}, {'agent':'planner','kind':'proposal','key':'budget','value':1200}, {'agent':'finance','kind':'fact','key':'budget','value':800}, {'agent':'manager','kind':'approval','key':'budget','value':800}]  # 构造六条跨 Agent 共享的事实、推测和审批事件。
print('黑板事件:', events)  # 输出原始事件流。
print('教学说明：fact 代表服务端或审批系统事实，proposal 只是模型建议。')  # 明确不同信息层的可信度。

黑板事件: [{'agent': 'planner', 'kind': 'proposal', 'key': 'city', 'value': '东京'}, {'agent': 'finance', 'kind': 'fact', 'key': 'budget', 'value': 800}, {'agent': 'compliance', 'kind': 'fact', 'key': 'policy', 'value': '经济舱'}, {'agent': 'planner', 'kind': 'proposal', 'key': 'budget', 'value': 1200}, {'agent': 'finance', 'kind': 'fact', 'key': 'budget', 'value': 800}, {'agent': 'manager', 'kind': 'approval', 'key': 'budget', 'value': 800}]
教学说明：fact 代表服务端或审批系统事实，proposal 只是模型建议。


In [2]:
unsafe_board = {}  # 初始化可被任意写者覆盖的错误共享字典。
for event in events:  # 按事件到达顺序写入同一个键空间。
    unsafe_board[event['key']] = event['value']  # 忽略 owner、kind 和版本，直接覆盖旧值。
print('最后写入基线:', unsafe_board)  # 输出无法判断预算来自事实还是推测的最终字典。
print('基线问题：历史与所有权消失后，无法审计谁改变了预算。')  # 解释覆盖式状态的不可审计性。

最后写入基线: {'city': '东京', 'budget': 800, 'policy': '经济舱'}
基线问题：历史与所有权消失后，无法审计谁改变了预算。


In [3]:
board = []  # 初始化 append-only 黑板事件表。
def publish(event):  # 定义带版本和写者身份的受控发布操作。
    item = dict(event)  # 复制输入事件避免调用方后续修改影响账本。
    item['version'] = len(board) + 1  # 为每次发布分配单调递增版本。
    board.append(item)  # 追加保存而不覆盖历史。
    return item  # 返回已落账的结构化 artifact。
def latest_fact(key):  # 定义只读取某字段最新权威事实的视图。
    facts = [item for item in board if item['key'] == key and item['kind'] in {'fact','approval'}]  # 过滤模型建议，保留事实和审批证据。
    return facts[-1] if facts else None  # 返回最后一个有权威语义的版本。

In [4]:
published = [publish(event) for event in events]  # 将六条事件依次写入 append-only 黑板。
budget_fact = latest_fact('budget')  # 从黑板获取预算字段的权威版本。
print('version | agent | kind | key | value')  # 输出可审计黑板表标题。
for item in published:  # 遍历保留全部历史的事件账本。
    print(item['version'], item['agent'], item['kind'], item['key'], item['value'])  # 输出写者、信息层和版本。
print('权威预算视图:', budget_fact)  # 输出与模型 proposal 分离后的可信预算。

version | agent | kind | key | value
1 planner proposal city 东京
2 finance fact budget 800
3 compliance fact policy 经济舱
4 planner proposal budget 1200
5 finance fact budget 800
6 manager approval budget 800
权威预算视图: {'agent': 'manager', 'kind': 'approval', 'key': 'budget', 'value': 800, 'version': 6}


In [5]:
proposal_budget = next(item for item in board if item['agent'] == 'planner' and item['key'] == 'budget')  # 取出规划 Agent 的非权威预算建议。
print('失败案例：proposal=', proposal_budget, '，事实视图=', budget_fact)  # 展示建议不会覆盖财务事实。
print('生产差距：需补充租户 ACL、敏感字段裁剪、artifact hash、TTL、事件签名和跨会话持久化。')  # 说明黑板教学实现的生产缺口。

失败案例：proposal= {'agent': 'planner', 'kind': 'proposal', 'key': 'budget', 'value': 1200, 'version': 4} ，事实视图= {'agent': 'manager', 'kind': 'approval', 'key': 'budget', 'value': 800, 'version': 6}
生产差距：需补充租户 ACL、敏感字段裁剪、artifact hash、TTL、事件签名和跨会话持久化。


In [6]:
assert budget_fact['value'] == 800  # 验证权威视图不会被 1200 的模型建议污染。
assert len(board) == 6  # 验证黑板保留了全部六条历史事件。
assert proposal_budget['kind'] == 'proposal'  # 验证模型建议与事实拥有不同信息层。